## 8-1 try...except...结构
如果用户输入的不是数字，产生ValueError异常

In [1]:
while True:
    try:
        x = int(input("Please enter a number: "))
        break
    except ValueError:
        print("This is not a valid number. Please try again...")


Please enter a number:  a


This is not a valid number. Please try again...


Please enter a number:  1


In [2]:
try:
    # 尝试执行的代码，可能会产生异常
    result = 10 / 0
except ZeroDivisionError:
    # 处理除零异常
    print("不能除以零。")
except TypeError:
    # 处理类型错误异常
    print("类型错误。")
else:
    # 如果没有异常发生
    print("操作成功。")
finally:
    # 无论是否发生异常
    print("操作完成。")


不能除以零。
操作完成。


## 8-2 自定义异常类

In [3]:
class MyCustomError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(message)

def check_age(age):
    if age < 18:
        raise MyCustomError("年龄不足18岁")
    return "年龄已确认"
try:
    user_age = check_age(16)
except MyCustomError as e:
    print("发生异常：", e)


发生异常： 年龄不足18岁


## 8-3 断言

In [4]:
def get_first_element(data_list):
    assert len(data_list) > 0, "The list cannot be empty!"
    return data_list[0]
data = [1, 2, 3, 4, 5]
try:
    print("First element:", get_first_element(data))
except AssertionError as error:
    print("Error:", error)
try:
    empty_data = []
    print("First element:", get_first_element(empty_data))
except AssertionError as error:
    print("Error:", error)


First element: 1
Error: The list cannot be empty!


## 8-4 上下文管理
计算分数列表中的最高分、最低分和平均分。

In [5]:
import sqlite3

# 假设数据库文件为 'database.db'
try:
    with sqlite3.connect('database.db') as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM some_table')
        rows = cursor.fetchall()
        if rows:
            for row in rows:
                print(row)
        else:
            print("查询结果为空。")
except sqlite3.Error as e:
    print(f"数据库错误: {e}")
except Exception as e:
    print(f"其他错误: {e}")


数据库错误: no such table: some_table


## 案例1：从文件中读取指定行数的文本内容
首先将指定的文本内容写入文件，然后根据用户输入，从文件中读取指定行数。如果用户输入的行数大于文件中的行数，程序将读取并输出文件的所有内容。该程序还需要处理可能的异常情况，包括文件写入时的 I/O 错误、文件不存在错误，以及用户输入无效数字的情况。

In [6]:
def write_to_file(file_path, file_content):
    try:
        with open(file_path, "w") as file:
            file.write("\n".join(file_content))
        print(f"已成功将内容写入文件 '{file_path}'。")
    except IOError as e:
        print(f"错误：无法写入文件 '{file_path}'。发生了 I/O 错误：{e}")
    except Exception as e:
        print(f"发生了未知错误：{e}")

def read_file_lines(file_path, num_lines):
    try:
        with open(file_path, "r") as file:
            lines = file.readlines()
        if num_lines > len(lines):
            num_lines = len(lines)
            print(f"文件只有 {len(lines)} 行，显示所有内容：")
        else:
            print(f"文件的前 {num_lines} 行内容如下：")
        for i in range(num_lines):
            print(lines[i].strip())
    except FileNotFoundError:
        print(f"错误：文件 '{file_path}' 不存在。")
    except Exception as e:
        print(f"发生了一个未知错误：{e}")

file_path = "example.txt"
file_content = [
    "空山新雨后，天气晚来秋。",
    "明月松间照，清泉石上流。",
    "竹喧归浣女，莲动下渔舟。",
    "随意春芳歇，王孙自可留。"
]

write_to_file(file_path, file_content)

try:
    num_lines = int(input("请输入要读取的行数："))
    read_file_lines(file_path, num_lines)
except ValueError:
    print("错误：请输入有效的数字。")


已成功将内容写入文件 'example.txt'。


请输入要读取的行数： 2


文件的前 2 行内容如下：
空山新雨后，天气晚来秋。
明月松间照，清泉石上流。


请输入要读取的行数： 4


文件的前 4 行内容如下：
空山新雨后，天气晚来秋。
明月松间照，清泉石上流。
竹喧归浣女，莲动下渔舟。
随意春芳歇，王孙自可留。


## 案例2:模拟一个银行账户的取款操作
使用异常处理来防止取款金额超过账户余额、无效的取款金额（例如负数或非数字值）和数据库操作异常（如果涉及到数据库交互）等情况。

In [7]:
class InvalidAmountError(Exception):
    """自定义异常：无效的取款金额"""
    pass


class DatabaseError(Exception):
    """自定义异常：数据库操作异常"""
    pass


class InsufficientFundsError(Exception):
    """自定义异常：余额不足"""
    pass


class BankAccount:
    def __init__(self, initial_balance):
        self.balance = initial_balance

    def withdraw(self, amount):
        # 检查是否为有效金额
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise InvalidAmountError(f'无效的取款金额：{amount}')

        # 模拟数据库操作异常
        if amount == 12345:
            raise DatabaseError("数据库操作异常")

        # 检查是否余额充足
        if amount > self.balance:
            raise InsufficientFundsError(f'账户余额不足，当前余额为：{self.balance}，尝试取款：{amount}')

        self.balance -= amount
        return self.balance


# 创建一个银行账户
account = BankAccount(10000)


# 尝试取款操作
def attempt_withdrawal(amount):
    try:
        print(f"尝试取款：{amount}")
        new_balance = account.withdraw(amount)
        print(f"取款成功，剩余余额：{new_balance}")
    except InsufficientFundsError as e:
        print("发生错误：", e)
    except InvalidAmountError as e:
        print("发生错误：", e)
    except DatabaseError as e:
        print("发生错误：", e)


# 测试不同的取款情况
attempt_withdrawal(15000)  # 超出余额
attempt_withdrawal(-100)  # 无效金额
attempt_withdrawal('abc')  # 无效金额
attempt_withdrawal(12345)  # 数据库异常
attempt_withdrawal(5000)  # 正常取款


尝试取款：15000
发生错误： 账户余额不足，当前余额为：10000，尝试取款：15000
尝试取款：-100
发生错误： 无效的取款金额：-100
尝试取款：abc
发生错误： 无效的取款金额：abc
尝试取款：12345
发生错误： 数据库操作异常
尝试取款：5000
取款成功，剩余余额：5000
